In [2]:
# ============================================================
# CORRECTED MIXMATCH ON MNIST
# 90/10 TO 10/90 LABELED / UNLABELED SPLITS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
import random
import gc


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

BATCH_SIZE = 128

# Keep 3 for runtime.
# For final paper run, change to 5 if required.
EPOCHS_INITIAL = 3
EPOCHS_MIXMATCH = 3

# MixMatch parameters
T = 0.5
ALPHA = 0.75
LAMBDA_U = 1.0


# ============================================================
# 2. MNIST DATASET
# ============================================================

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print(
    "Total training samples:",
    len(train_dataset)
)

print(
    "Total test samples:",
    len(test_dataset)
)


# ============================================================
# 3. CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self):

        super(CNN, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 7 * 7,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                10
            )
        )

    def forward(self, x):

        x = self.conv(x)

        x = self.fc(x)

        return x


# ============================================================
# 4. TEST ACCURACY
# ============================================================

def test_accuracy(model, test_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        100.0 * correct / total
    )


# ============================================================
# 5. INITIAL SUPERVISED TRAINING
# ============================================================

def supervised_train(
    model,
    labeled_loader,
    epochs
):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for images, labels in labeled_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                *
                images.size(0)
            )

        avg_loss = (
            total_loss /
            len(labeled_loader.dataset)
        )

        print(
            f"      Epoch "
            f"{epoch + 1}/{epochs} "
            f"Loss: {avg_loss:.4f}"
        )

    return model, optimizer


# ============================================================
# 6. MIXMATCH TRAINING
# ============================================================

def mixmatch_train(
    model,
    labeled_loader,
    unlabeled_loader,
    epochs,
    T=0.5,
    alpha=0.75,
    lambda_u=1.0
):

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0
        total_supervised = 0.0
        total_unsupervised = 0.0

        unlabeled_iter = iter(
            unlabeled_loader
        )

        for images_l, labels_l in labeled_loader:

            # =================================================
            # LABELED DATA
            # =================================================

            images_l = images_l.to(
                device,
                non_blocking=True
            )

            labels_l = labels_l.to(
                device,
                non_blocking=True
            )

            # =================================================
            # GET UNLABELED BATCH
            # =================================================

            try:

                images_u, _ = next(
                    unlabeled_iter
                )

            except StopIteration:

                unlabeled_iter = iter(
                    unlabeled_loader
                )

                images_u, _ = next(
                    unlabeled_iter
                )

            images_u = images_u.to(
                device,
                non_blocking=True
            )

            # =================================================
            # GENERATE SOFT PSEUDO-LABELS
            # =================================================

            with torch.no_grad():

                outputs_u = model(
                    images_u
                )

                probs_u = torch.softmax(
                    outputs_u,
                    dim=1
                )

                # =================================================
                # SHARPENING
                #
                # p' = p^(1/T) / sum(p^(1/T))
                # =================================================

                probs_u = (
                    probs_u **
                    (1.0 / T)
                )

                probs_u = (
                    probs_u /
                    probs_u.sum(
                        dim=1,
                        keepdim=True
                    )
                )

            # =================================================
            # ONE-HOT LABELS
            # =================================================

            labels_onehot = torch.zeros(
                labels_l.size(0),
                10,
                device=device
            )

            labels_onehot.scatter_(
                1,
                labels_l.unsqueeze(1),
                1.0
            )

            # =================================================
            # COMBINE LABELED + UNLABELED
            # =================================================

            all_images = torch.cat(
                [
                    images_l,
                    images_u
                ],
                dim=0
            )

            all_targets = torch.cat(
                [
                    labels_onehot,
                    probs_u
                ],
                dim=0
            )

            # =================================================
            # MIXUP
            # =================================================

            lam = np.random.beta(
                alpha,
                alpha
            )

            # Keep lambda at least 0.5
            # so that the primary sample has
            # greater/equal contribution.
            lam = max(
                lam,
                1.0 - lam
            )

            index = torch.randperm(
                all_images.size(0),
                device=device
            )

            mixed_images = (
                lam * all_images
                +
                (1.0 - lam)
                *
                all_images[index]
            )

            mixed_targets = (
                lam * all_targets
                +
                (1.0 - lam)
                *
                all_targets[index]
            )

            # =================================================
            # FORWARD PASS
            # =================================================

            outputs = model(
                mixed_images
            )

            # Convert logits to probabilities
            predictions = torch.softmax(
                outputs,
                dim=1
            )

            # =================================================
            # SEPARATE LABELED AND UNLABELED PARTS
            # =================================================

            batch_labeled = (
                images_l.size(0)
            )

            predictions_l = (
                predictions[
                    :batch_labeled
                ]
            )

            predictions_u = (
                predictions[
                    batch_labeled:
                ]
            )

            targets_l = (
                mixed_targets[
                    :batch_labeled
                ]
            )

            targets_u = (
                mixed_targets[
                    batch_labeled:
                ]
            )

            # =================================================
            # SUPERVISED LOSS
            #
            # Soft-target cross entropy
            # =================================================

            supervised_loss = -torch.mean(
                torch.sum(
                    targets_l
                    *
                    torch.log(
                        predictions_l
                        + 1e-8
                    ),
                    dim=1
                )
            )

            # =================================================
            # UNSUPERVISED CONSISTENCY LOSS
            #
            # MSE between predicted probabilities
            # and soft pseudo-labels
            # =================================================

            unsupervised_loss = torch.mean(
                (
                    predictions_u
                    -
                    targets_u
                ) ** 2
            )

            # =================================================
            # TOTAL LOSS
            # =================================================

            loss = (
                supervised_loss
                +
                lambda_u
                *
                unsupervised_loss
            )

            # =================================================
            # BACKPROPAGATION
            # =================================================

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            # =================================================
            # TRACK LOSSES
            # =================================================

            total_loss += loss.item()

            total_supervised += (
                supervised_loss.item()
            )

            total_unsupervised += (
                unsupervised_loss.item()
            )

        num_batches = len(
            labeled_loader
        )

        print(
            f"      Epoch "
            f"{epoch + 1}/{epochs} "
            f"Total Loss: "
            f"{total_loss / num_batches:.4f} | "
            f"Supervised: "
            f"{total_supervised / num_batches:.4f} | "
            f"Unsupervised: "
            f"{total_unsupervised / num_batches:.4f}"
        )

    return model


# ============================================================
# 7. TEST LOADER
# ============================================================

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        device.type == "cuda"
    )
)


# ============================================================
# 8. NINE EXPERIMENTAL SPLITS
# ============================================================

split_fractions = [

    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
    0.40,
    0.30,
    0.20,
    0.10

]


# ============================================================
# 9. RESULTS
# ============================================================

all_results = []

total_samples = len(
    train_dataset
)


# ============================================================
# 10. RUN ALL SPLITS
# ============================================================

for labeled_fraction in split_fractions:

    # ========================================================
    # SPLIT NAME
    # ========================================================

    labeled_percent = int(
        labeled_fraction * 100
    )

    unlabeled_percent = (
        100 -
        labeled_percent
    )

    split_name = (
        f"{labeled_percent}/"
        f"{unlabeled_percent}"
    )

    print("\n" + "=" * 70)

    print(
        f"EXPERIMENT: "
        f"{split_name} "
        f"(LABELED / UNLABELED)"
    )

    print("=" * 70)


    # ========================================================
    # CREATE RANDOM SPLIT
    # ========================================================

    rng = np.random.default_rng(
        SEED
    )

    indices = np.arange(
        total_samples
    )

    rng.shuffle(indices)

    num_labeled = int(
        total_samples *
        labeled_fraction
    )

    labeled_idx = (
        indices[
            :num_labeled
        ]
    )

    unlabeled_idx = (
        indices[
            num_labeled:
        ]
    )

    print(
        f"Initial labeled samples: "
        f"{len(labeled_idx)}"
    )

    print(
        f"Initial unlabeled samples: "
        f"{len(unlabeled_idx)}"
    )


    # ========================================================
    # CREATE DATASETS
    # ========================================================

    labeled_dataset = Subset(
        train_dataset,
        labeled_idx
    )

    unlabeled_dataset = Subset(
        train_dataset,
        unlabeled_idx
    )


    # ========================================================
    # CREATE DATALOADERS
    # ========================================================

    labeled_loader = DataLoader(

        labeled_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )

    unlabeled_loader = DataLoader(

        unlabeled_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )


    # ========================================================
    # NEW MODEL FOR EACH SPLIT
    # ========================================================

    model = CNN().to(device)


    # ========================================================
    # INITIAL SUPERVISED TRAINING
    # ========================================================

    print(
        "\nInitial supervised training..."
    )

    model, _ = supervised_train(

        model,

        labeled_loader,

        EPOCHS_INITIAL

    )


    # ========================================================
    # INITIAL ACCURACY
    # ========================================================

    initial_accuracy = test_accuracy(
        model,
        test_loader
    )

    print(
        f"Initial Test Accuracy: "
        f"{initial_accuracy:.2f}%"
    )


    # ========================================================
    # SAVE INITIAL RESULT
    # ========================================================

    all_results.append({

        "split":
            split_name,

        "stage":
            "Initial",

        "accuracy_percent":
            round(
                initial_accuracy,
                2
            )

    })


    # ========================================================
    # MIXMATCH
    # ========================================================

    print(
        "\n--- Corrected MixMatch Training ---"
    )

    print(
        f"Temperature T = {T}"
    )

    print(
        f"MixUp alpha = {ALPHA}"
    )

    print(
        f"Unsupervised weight = {LAMBDA_U}"
    )


    model = mixmatch_train(

        model,

        labeled_loader,

        unlabeled_loader,

        EPOCHS_MIXMATCH,

        T=T,

        alpha=ALPHA,

        lambda_u=LAMBDA_U

    )


    # ========================================================
    # FINAL MIXMATCH ACCURACY
    # ========================================================

    final_accuracy = test_accuracy(
        model,
        test_loader
    )

    print(
        f"MixMatch Test Accuracy: "
        f"{final_accuracy:.2f}%"
    )


    # ========================================================
    # SAVE FINAL RESULT
    # ========================================================

    all_results.append({

        "split":
            split_name,

        "stage":
            "MixMatch",

        "accuracy_percent":
            round(
                final_accuracy,
                2
            )

    })


    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 11. FINAL RESULTS
# ============================================================

mixmatch_results = pd.DataFrame(
    all_results
)

print("\n" + "=" * 70)

print(
    "FINAL CORRECTED MIXMATCH RESULTS"
)

print("=" * 70)

display(
    mixmatch_results
)


# ============================================================
# 12. FINAL MIXMATCH ONLY
# ============================================================

final_mixmatch = (
    mixmatch_results[
        mixmatch_results["stage"]
        ==
        "MixMatch"
    ]
    .copy()
)

print(
    "\nFinal MixMatch accuracy "
    "for each split:"
)

display(
    final_mixmatch
)


# ============================================================
# 13. SAVE RESULTS
# ============================================================

mixmatch_results.to_csv(
    "mixmatch_corrected_results.csv",
    index=False
)

final_mixmatch.to_csv(
    "mixmatch_corrected_final_results.csv",
    index=False
)

print(
    "\nResults saved:"
)

print(
    "mixmatch_corrected_results.csv"
)

print(
    "mixmatch_corrected_final_results.csv"
)

Device: cuda
Total training samples: 60000
Total test samples: 10000

EXPERIMENT: 90/10 (LABELED / UNLABELED)
Initial labeled samples: 54000
Initial unlabeled samples: 6000

Initial supervised training...
      Epoch 1/3 Loss: 0.2652
      Epoch 2/3 Loss: 0.0643
      Epoch 3/3 Loss: 0.0442
Initial Test Accuracy: 98.86%

--- Corrected MixMatch Training ---
Temperature T = 0.5
MixUp alpha = 0.75
Unsupervised weight = 1.0
      Epoch 1/3 Total Loss: 0.7774 | Supervised: 0.7719 | Unsupervised: 0.0056
      Epoch 2/3 Total Loss: 0.6804 | Supervised: 0.6758 | Unsupervised: 0.0046
      Epoch 3/3 Total Loss: 0.6328 | Supervised: 0.6285 | Unsupervised: 0.0043
MixMatch Test Accuracy: 98.67%

EXPERIMENT: 80/20 (LABELED / UNLABELED)
Initial labeled samples: 48000
Initial unlabeled samples: 12000

Initial supervised training...
      Epoch 1/3 Loss: 0.2976
      Epoch 2/3 Loss: 0.0742
      Epoch 3/3 Loss: 0.0514
Initial Test Accuracy: 98.67%

--- Corrected MixMatch Training ---
Temperature T = 0

,split,stage,accuracy_percent
0,90/10,Initial,98.86
1,90/10,MixMatch,98.67
2,80/20,Initial,98.67
3,80/20,MixMatch,98.74
4,70/30,Initial,98.48
5,70/30,MixMatch,98.64
6,60/40,Initial,98.31
7,60/40,MixMatch,98.63
8,50/50,Initial,97.93
9,50/50,MixMatch,98.64



Final MixMatch accuracy for each split:


,split,stage,accuracy_percent
1,90/10,MixMatch,98.67
3,80/20,MixMatch,98.74
5,70/30,MixMatch,98.64
7,60/40,MixMatch,98.63
9,50/50,MixMatch,98.64
11,40/60,MixMatch,98.39
13,30/70,MixMatch,97.93
15,20/80,MixMatch,98.06
17,10/90,MixMatch,97.00



Results saved:
mixmatch_corrected_results.csv
mixmatch_corrected_final_results.csv
